# Prompt Chaining: Sequential Steps with Quality Gates

**Notebook 2 of 7 in the "Building Effective AI Agents" tutorial series.**

## What You'll Learn

- How to decompose complex tasks into sequential LLM steps
- How to add programmatic gates between steps to verify intermediate outputs
- When to use prompt chaining vs. other workflow patterns

## Where This Fits on the Complexity Spectrum

The simplest workflow pattern — a linear pipeline where each step builds on the last.

> "Prompt chaining decomposes a task into a sequence of steps, where each LLM call processes the output of the previous one."

## How It Works

![Prompt chaining workflow — sequential LLM calls with a gate check between steps](assets/prompt_chaining.webp)

**The flow:** Input → LLM Call 1 → Gate (pass/fail) → LLM Call 2 → LLM Call 3 → Output

**Key concepts:**
- Each step is a focused, single-purpose LLM call
- **Gates** are programmatic checks between steps that verify intermediate outputs
- If a gate fails, the process exits early rather than propagating errors
- The trade-off: increased latency for higher accuracy (each individual call is simpler)

## When to Use Prompt Chaining

**Use when:**
- The task can be cleanly decomposed into fixed, sequential subtasks
- You want to trade latency for higher accuracy
- You need programmatic verification between steps (e.g., format checks, safety filters)
- Each step's output is the next step's input

**Don't use when:**
- Subtasks are independent (use [Parallelization](03_parallelization.ipynb) instead)
- You can't predict the subtasks in advance (use [Orchestrator-Workers](04_orchestrator_workers.ipynb))
- A single well-crafted prompt handles the task adequately

In [ ]:
import sys
sys.path.append(".")
from util import llm_call, extract_xml

In [ ]:
def chain(input_text: str, prompts: list[str]) -> str:
    """Chain multiple LLM calls sequentially, passing results between steps."""
    result = input_text
    for i, prompt in enumerate(prompts, 1):
        print(f"\n{'─' * 50}")
        print(f"  Step {i}")
        print(f"{'─' * 50}")
        result = llm_call(f"{prompt}\nInput: {result}")
        print(result[:500])  # Preview first 500 chars
    return result

## Example 1: Data Extraction Pipeline

A classic prompt chaining use case — progressively transforming raw text through extraction, normalization, sorting, and formatting. Each step is simple enough for high accuracy.

In [ ]:
data_processing_steps = [
    """Extract only the numerical values and their associated metrics from the text.
    Format each as 'value: metric' on a new line.
    Example format:
    92: customer satisfaction
    45%: revenue growth""",

    """Convert all numerical values to percentages where possible.
    If not a percentage or points, convert to decimal (e.g., 92 points -> 92%).
    Keep one number per line.""",

    """Sort all lines in descending order by numerical value.
    Keep the format 'value: metric' on each line.""",

    """Format the sorted data as a markdown table with columns:
    | Metric | Value |
    |:--|--:|"""
]

report = """
Q3 Performance Summary:
Our customer satisfaction score rose to 92 points this quarter.
Revenue grew by 45% compared to last year.
Market share is now at 23% in our primary market.
Customer churn decreased to 5% from 8%.
New user acquisition cost is $43 per user.
Product adoption rate increased to 78%.
Employee satisfaction is at 87 points.
Operating margin improved to 34%.
"""

print("Input text:")
print(report)
formatted_result = chain(report, data_processing_steps)
print(f"\n{'═' * 50}")
print("  FINAL OUTPUT")
print(f"{'═' * 50}")
print(formatted_result)

## Adding Gates: Programmatic Verification

Gates are what make prompt chaining robust. They're simple checks between steps that catch problems early:

| Gate Type | Example | Action on Failure |
|-----------|---------|-------------------|
| **Format validation** | Does output contain required fields? | Exit with error |
| **Content check** | Is the output within expected length? | Retry the step |
| **Safety filter** | Does output contain prohibited content? | Exit with warning |
| **Quality threshold** | Does output meet minimum criteria? | Retry or fallback |

Gates transform a fragile chain into a reliable pipeline.

In [ ]:
def chain_with_gate(input_text: str, prompts: list[str], gate_fn=None, gate_after_step: int = 1) -> str:
    """Chain with a programmatic gate check between specified steps."""
    result = input_text
    for i, prompt in enumerate(prompts, 1):
        print(f"\n{'─' * 50}")
        print(f"  Step {i}")
        print(f"{'─' * 50}")
        result = llm_call(f"{prompt}\nInput: {result}")
        print(result[:300])
        
        # Apply gate after specified step
        if gate_fn and i == gate_after_step:
            passed, reason = gate_fn(result)
            if passed:
                print(f"\n  \u2713 Gate passed: {reason}")
            else:
                print(f"\n  \u2717 Gate FAILED: {reason}")
                print("  \u2192 Exiting chain early to avoid propagating errors.")
                return None
    return result


# Gate function: verify outline quality before writing full document
def outline_gate(outline: str) -> tuple[bool, str]:
    """Check that an outline has sufficient structure before proceeding."""
    lines = [l.strip() for l in outline.split('\n') if l.strip()]
    
    has_intro = any('intro' in l.lower() or '1.' in l for l in lines)
    has_conclusion = any('conclu' in l.lower() or 'summary' in l.lower() for l in lines)
    section_count = sum(1 for l in lines if l.startswith(('#', '-', '*')) or l[0:1].isdigit())
    
    if section_count < 3:
        return False, f"Only {section_count} sections found (need at least 3)"
    if not has_intro:
        return False, "No introduction section detected"
    return True, f"Outline has {section_count} sections with proper structure"


# Document authoring pipeline with gate
authoring_steps = [
    """Create a detailed outline for a technical blog post about the given topic.
    Include an introduction, at least 3 main sections with subsections, and a conclusion.
    Format with markdown headers and bullet points.""",

    """Using the outline provided, write the full blog post.
    Each section should be 2-3 paragraphs.
    Use clear, technical language appropriate for software engineers.""",

    """Edit the blog post for clarity and conciseness.
    Remove redundancy, tighten language, and ensure logical flow.
    Add a TL;DR at the top."""
]

topic = "Why prompt chaining is more reliable than single complex prompts"
result = chain_with_gate(topic, authoring_steps, gate_fn=outline_gate, gate_after_step=1)

## Pitfalls & Common Mistakes

| Pitfall | What goes wrong | How to avoid |
|---------|----------------|---------------|
| **Over-decomposition** | Too many tiny steps add latency without improving quality | Each step should handle a meaningful transformation |
| **Strict gates** | Legitimate outputs get rejected, blocking progress | Tune gate thresholds with real examples |
| **Lenient gates** | Bad outputs propagate and corrupt downstream steps | Test gates with known-bad inputs |
| **No error context** | When a chain fails, you don't know which step broke | Log intermediate outputs at each step |
| **Tight coupling** | Changing one step breaks downstream steps | Define clear contracts (expected format) between steps |

### When to Graduate to Other Patterns

- If steps are **independent** → [Parallelization](03_parallelization.ipynb)
- If steps are **unpredictable** → [Orchestrator-Workers](04_orchestrator_workers.ipynb)
- If you need **iterative refinement** → [Evaluator-Optimizer](05_evaluator_optimizer.ipynb)

## Key Takeaways

1. **Prompt chaining = sequential LLM calls + programmatic gates** — the simplest workflow pattern
2. **Gates are the safety net** — they catch errors early before they propagate
3. **Each step should be simpler than the whole** — that's the entire value proposition
4. **Trade-off is latency vs. accuracy** — N simple calls often outperform 1 complex call

---

**Next up:** [02_routing.ipynb](02_routing.ipynb) — Learn to classify inputs and dispatch to specialized handlers.